# Muon vs SGD: Staged Linear Network

Consider a two-layer linear network $\hat y = W_2 W_1 x$, with the target $W_2 W_1 \approx A$.

**Target decomposition.** Take the SVD of $A$: $A = U \Sigma V^\top$, and define
$$W_1^* = \sqrt{\Sigma}\, V^\top, \quad W_2^* = U \sqrt{\Sigma}$$
so that $W_2^* W_1^* = A$. In Phase 1, only $W_1$ is trained, using $\|W_1 - W_1^*\|$ as the loss. In Phase 2, only $W_2$ is trained, using the composition error $\|W_2 W_1 - A\|$ as the loss.

**Staged training**
- Phase 1 (SGD vs Muon): freeze $W_2$ and train $W_1$ with either SGD or Muon, minimizing $\|W_1 - W_1^*\|^2 / \|A\|^2$.
- Phase 2 (SGD only): freeze the $W_1$ obtained from Phase 1, reinitialize $W_2$, and use only SGD to minimize $\|W_2 W_1 - A\|^2 / \|A\|^2$ until $\|W_2 W_1 - A\| / \|A\| \le \text{THRESHOLD}$.

Muon uses the same learning rate as SGD. At each step, it applies NS orthogonalization to the current gradient, with update norm lr $\|g\|$.

In [ ]:
import math

import torch

torch.set_default_dtype(torch.float64)

D = 64
PHASE1_LR = 2e-1
PHASE2_LR = 2e-2
SEED = 0
THRESHOLD = 0.05
PHASE1_STEPS = 600
PHASE2_MAX_STEPS = 20000

In [ ]:
def zeropower_via_newtonschulz5(G, steps=3, eps=1e-7):
    assert len(G.shape) == 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.bfloat16()
    X = X / (X.norm() + eps)
    if G.size(0) > G.size(1):
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    if G.size(0) > G.size(1):
        X = X.T
    return X.to(dtype=G.dtype)


def muon_direction(g, eps=1e-7):
    direction = zeropower_via_newtonschulz5(g)
    return direction * (g.norm() / (direction.norm() + eps))


class Muon(torch.optim.Optimizer):

    def __init__(self, params, lr=1e-3):
        super().__init__(params, dict(lr=lr))

    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            for p in group["params"]:
                g = p.grad
                if g is None:
                    continue
                p.data.add_(muon_direction(g), alpha=-lr)

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_target(d, seed=0):
    gen = torch.Generator().manual_seed(seed)
    return torch.randn(d, d, generator=gen) / math.sqrt(d)


def svd_factors(A):
    u, s, vh = torch.linalg.svd(A.cpu(), full_matrices=False)
    sqrt_s = torch.sqrt(s)
    w1_star = (torch.diag(sqrt_s) @ vh).cuda()
    w2_star = (u @ torch.diag(sqrt_s)).cuda()
    return w1_star, w2_star


def rel_dist(M, target, scale):
    return (M - target).norm().item() / (scale + 1e-12)


def compose_loss(W2, W1, A, norm_A):
    M = W2 @ W1 - A
    return (M ** 2).sum() / (norm_A**2)


def steps_to(values, threshold):
    for i, v in enumerate(values):
        if v <= threshold:
            return i
    return None


def rand_matrix(d, seed):
    state = torch.cuda.get_rng_state()
    torch.cuda.manual_seed(seed)
    W = torch.randn(d, d, device="cuda") / math.sqrt(d)
    torch.cuda.set_rng_state(state)
    return W


def w1_spectrum_flatness(W1):
    s = torch.linalg.svdvals(W1)
    return (s.pow(2).sum() / s[0].pow(2)).item() / D

In [ ]:
def train_staged(
    A,
    w1_star,
    w2_star,
    phase1_optimizer_class,
    phase1_steps=PHASE1_STEPS,
    phase2_max_steps=PHASE2_MAX_STEPS,
    phase2_threshold=THRESHOLD,
    phase2_lr=PHASE2_LR,
    seed=SEED,
):
    set_seed(seed)
    W1 = rand_matrix(D, seed)
    norm_A = A.norm().item()
    w1_star = w1_star.detach()

    w1_dist = []

    W1.requires_grad_(True)
    opt1 = phase1_optimizer_class([W1], lr=PHASE1_LR)
    for step in range(phase1_steps + 1):
        w1_dist.append(rel_dist(W1.detach(), w1_star, norm_A))
        if step == phase1_steps:
            break
        loss = ((W1 - w1_star) ** 2).sum() / (norm_A**2)
        opt1.zero_grad(set_to_none=True)
        loss.backward()
        opt1.step()

    W1_frozen = W1.detach().clone()

    W2 = rand_matrix(D, seed + 1)
    W2.requires_grad_(True)
    opt2 = torch.optim.SGD([W2], lr=phase2_lr)
    tot_dist, w2_dist = [], []
    phase2_steps = 0
    for step in range(phase2_max_steps + 1):
        W2d = W2.detach()
        dist = rel_dist(W2d @ W1_frozen, A, norm_A)
        tot_dist.append(dist)
        w2_dist.append(rel_dist(W2d, w2_star, norm_A))
        phase2_steps = step
        if dist <= phase2_threshold:
            break
        if step == phase2_max_steps:
            break
        loss = compose_loss(W2, W1_frozen, A, norm_A)
        opt2.zero_grad(set_to_none=True)
        loss.backward()
        opt2.step()

    return dict(
        tot_dist=tot_dist,
        w1_dist=w1_dist,
        w2_dist=w2_dist,
        W1=W1_frozen,
        W2=W2.detach(),
        phase1_steps=phase1_steps,
        phase2_steps=phase2_steps,
    )


def run_staged_experiment(A, w1_star, w2_star):
    print("\n" + "=" * 72)
    print(f"Staged training: phase1 ({PHASE1_STEPS} steps), phase2 (threshold {THRESHOLD})")
    print(f"  phase1 lr={PHASE1_LR}, loss = ||W1-W1*||^2/||A||^2")
    print(f"  phase2 lr={PHASE2_LR}, loss = ||W2 W1 - A||^2/||A||^2")
    print("=" * 72)
    sgd = train_staged(A, w1_star, w2_star, torch.optim.SGD)
    muon = train_staged(A, w1_star, w2_star, Muon)
    norm_A = A.norm().item()

    for label, run in [("SGD W1 -> SGD W2", sgd), ("Muon W1 -> SGD W2", muon)]:
        p1 = run["phase1_steps"]
        print(f"\n  [{label}]")
        print(f"    phase1 step {p1}: ||W1-W1*||={run['w1_dist'][-1]:.4f}")
        print(f"    phase2 step {p1 + run['phase2_steps']}: ||W1-W1*||={rel_dist(run['W1'], w1_star, norm_A):.4f}, "
              f"||W2-W2*||={run['w2_dist'][-1]:.4f}, ||W2 W1 - A||/||A||={run['tot_dist'][-1]:.6f}, "
              f"steps={run['phase2_steps']}")
        p2_hit = steps_to(run["tot_dist"], THRESHOLD)
        total = p1 + p2_hit if p2_hit is not None else None
        print(f"    steps@{THRESHOLD} total={total}")

    sgd_p2 = steps_to(sgd["tot_dist"], THRESHOLD)
    muon_p2 = steps_to(muon["tot_dist"], THRESHOLD)
    delta = sgd_p2 - muon_p2 if sgd_p2 and muon_p2 else None
    print(f"\n  delta={delta} (positive => Muon faster)")

    print(f"\n  Phase1 end W1 spectrum flatness (higher => flatter):")
    for label, run in [("SGD W1", sgd), ("Muon W1", muon)]:
        flat = w1_spectrum_flatness(run["W1"])
        print(f"    {label}: {flat:.4f}")
    flat_s = w1_spectrum_flatness(sgd["W1"])
    flat_m = w1_spectrum_flatness(muon["W1"])
    print(f"    Muon - SGD: {flat_m - flat_s:+.4f}")

    return sgd, muon

In [ ]:
set_seed(SEED)
A = make_target(D, seed=SEED).cuda()
w1_star, w2_star = svd_factors(A)
sgd_run, muon_run = run_staged_experiment(A, w1_star, w2_star)

## Summary

Muon ends Phase 1 farther from $W_1^*$, but the spectrum of its $W_1$ is more favorable for Phase 2. Net effect: Phase 2 may require fewer steps, and the overall optimization process can still be faster (`delta > 0`).

Why can Muon be faster in Phase 2?

Phase 2 is equivalent to fixing the encoder $W_1$ and learning a linear probe $W_2$ such that $W_2 W_1 \approx A$. Let $E_t = W_{2,t}W_1 - A$. Then
$$L(W_2)=\tfrac12\|W_2 W_1 - A\|_F^2,\qquad \nabla_{W_2}L=(W_2 W_1 - A)W_1^\top$$
The SGD update is $W_{2,t+1}=W_{2,t}-\eta(W_{2,t}W_1-A)W_1^\top$, which gives the error recurrence
$$E_{t+1}=E_t(I-\eta W_1^\top W_1)$$
In the eigenbasis of $W_1^\top W_1$, the $i$-th singular value $s_i(E_t)$ of $E_t$ satisfies $s_i(E_{t+1})=|1-\eta\sigma_i^2|\,s_i(E_t)$, where $\sigma_i$ is the $i$-th singular value of $W_1$. When $\eta=\alpha/\sigma_{\max}(W_1)^2$, the slowest contraction factor is $1-\alpha/\kappa(W_1)^2$.

Therefore, the closer $W_1$ is to orthogonal (flatter spectrum, smaller condition number), the faster Phase 2 optimizes.

Above, the loss is constructed directly from $W_2 W_1 - A$. In practice, a more common setup is to sample $x$, compute the output $y=W_2 W_1 x$, and compare it with the target $Ax$.

One can show that the closer $W_1$ is to orthogonal, the easier it is for a linear probe to recover the original input $x$ from the intermediate representation $h=W_1 x$. This is the same reason that learning $Ax$ becomes easier. To recover $x$, learn a probe $B$ such that $B W_1 \approx I$, with loss
$$L_x(B)=\tfrac12\|B W_1 - I\|_F^2$$
To recover $Ax$, simply replace the target matrix with $A$:
$$L_A(B)=\tfrac12\|B W_1 - A\|_F^2$$
Let $E_t = B_t W_1 - M$ ($M=I$ or $M=A$). We again have $E_{t+1} = E_t(I - \eta W_1^\top W_1)$, so the convergence speed is still determined only by the singular values of $W_1$.

## Interpreting Muon

In summary, Muon's advantage can be understood as follows. Each upstream update plays two roles at once: first, it reduces the current loss along the negative-gradient direction so training can continue; second, it acts like a temporary residual connection, preserving as much input information as possible in the output and passing it downstream. Because it is not known in advance whether the downstream module will need $x$ or $W_1 x$ more, encoding $x$ through an orthogonal transformation is the most favorable choice for downstream linear recovery of $x$. Muon's trade-off is exactly to orthogonalize the gradient before using it as the update direction. For the Phase 1 subproblem, this is no longer the steepest descent direction under that loss. But the resulting $W_1$ provides a better representation for the downstream module, making Phase 2 easier. Once Phase 2 optimizes better, it can in turn provide Phase 1 with an easier subproblem, compensating for Muon's weaker single-subproblem solving ability in Phase 1 and ultimately making the overall optimization faster.